# 高阶矩组合优化 (均值-方差-偏度-峰度)

**参考论文**: Qingfu Liu (2020), "The effectiveness of incorporating higher moments in portfolio strategies"  
**核心思想**: 将传统二阶矩(均值-方差)扩展至四阶矩(均值-方差-偏度-峰度)，多目标优化  
**创新**: 投资者不仅关注收益和风险，还偏好正偏度（右厚尾机会）和低峰度（避免极端事件）

> **⚠️ 教学演示声明**
> 
> 本 Notebook 为量化策略算法教学样例，回测结果包含**前视偏差 (Look-ahead Bias)**：
> - 训练标签使用了未来收益率（`shift(-N)`）
> - StandardScaler 等预处理可能在全量数据上拟合
> - 模型参数选择可能基于完整样本期
> 
> **所有回测收益率仅供学习参考，不代表策略的实际可期望表现，不可直接用于实盘交易。**

## 策略原理

### 四阶矩定义

设组合收益 $R_p = w^T r$:

**一阶矩 (均值)**: $\mu_p = E[R_p] = w^T \mu$

**二阶矩 (方差)**: $\sigma_p^2 = w^T \Sigma w$

**三阶矩 (偏度)**: $s_p = \frac{E[(R_p - \mu_p)^3]}{\sigma_p^3} = \frac{w^T S_3 (w \otimes w)}{\sigma_p^3}$

其中 $S_3$ 为三阶共偏度张量 (coskewness tensor, $N \times N^2$)

**四阶矩 (峰度)**: $k_p = \frac{E[(R_p - \mu_p)^4]}{\sigma_p^4} = \frac{w^T S_4 (w \otimes w \otimes w)}{\sigma_p^4}$

其中 $S_4$ 为四阶共峰度张量 (cokurtosis tensor, $N \times N^3$)

### 多目标优化

投资者效用函数 (扩展):
$$U(w) = \mu_p - \frac{\lambda_2}{2} \sigma_p^2 + \frac{\lambda_3}{6} s_p \sigma_p^3 - \frac{\lambda_4}{24} k_p \sigma_p^4$$

简化为加权目标:
$$\max_w \quad \alpha_1 \mu_p - \alpha_2 \sigma_p^2 + \alpha_3 s_p - \alpha_4 k_p$$

$$\text{s.t.} \quad \sum w_i = 1, \quad w_i \geq 0$$

### 直觉理解
- **正偏度** (skewness > 0): 收益分布右尾更长 → 大涨的概率更高
- **低峰度** (kurtosis): 减少极端事件发生的概率 (尾部更薄)
- 二阶矩模型假设正态分布 → 忽略了偏度和峰度信息

In [ ]:
# Cell 3: 动画 - 2-moment vs 4-moment 有效前沿对比
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

np.random.seed(42)

def animate_moment_frontiers():
    """对比2阶矩和4阶矩优化下的有效前沿差异"""
    n_assets = 8
    mu = np.array([0.06, 0.07, 0.12, 0.10, 0.15, 0.18, 0.14, 0.05])
    cov = np.eye(n_assets) * 0.06
    for i in range(n_assets):
        for j in range(i+1, n_assets):
            c = 0.015 + np.random.uniform(-0.005, 0.005)
            cov[i, j] = cov[j, i] = c

    n = 3000
    ret_2m, risk_2m, skew_2m = [], [], []
    ret_4m, risk_4m, skew_4m = [], [], []

    # 模拟非正态资产收益 (有偏度和峰度)
    asset_skewness = np.array([-0.3, -0.5, 0.8, 0.3, 1.0, -0.8, 0.5, -0.2])
    asset_kurtosis = np.array([3.5, 4.0, 5.0, 3.2, 6.0, 5.5, 4.0, 3.0])

    for _ in range(n):
        w = np.random.dirichlet(np.ones(n_assets))
        r = w @ mu
        s = np.sqrt(w @ cov @ w)

        # 2-moment: 纯MV
        ret_2m.append(r); risk_2m.append(s)
        skew_2m.append(w @ asset_skewness)

        # 4-moment: 带偏度/峰度偏好的权重调整
        skew_bonus = 0.02 * (w @ asset_skewness)  # 偏好正偏度
        kurt_penalty = 0.01 * (w @ asset_kurtosis - 3)  # 惩罚高峰度
        r_adj = r + skew_bonus - kurt_penalty
        ret_4m.append(r_adj); risk_4m.append(s)
        skew_4m.append(w @ asset_skewness)

    frames = []
    batch = 150
    for step in range(batch, n + 1, batch):
        frames.append(go.Frame(
            data=[
                go.Scatter(x=risk_2m[:step], y=ret_2m[:step], mode='markers',
                           name='2-Moment (MV)', marker=dict(color=skew_2m[:step],
                           colorscale='RdBu', cmin=-1, cmax=1, size=3, opacity=0.4,
                           colorbar=dict(title='偏度', x=1.02))),
                go.Scatter(x=risk_4m[:step], y=ret_4m[:step], mode='markers',
                           name='4-Moment (MVSK)', marker=dict(color=skew_4m[:step],
                           colorscale='Viridis', cmin=-1, cmax=1, size=3, opacity=0.4)),
            ],
            name=f'{step} portfolios'
        ))

    fig = go.Figure(data=frames[0].data, frames=frames)
    fig.update_layout(
        title='有效前沿: 2-Moment (MV) vs 4-Moment (MVSK)',
        xaxis_title='年化风险 (波动率)', yaxis_title='调整后年化收益',
        height=550, width=900,
        updatemenus=[dict(type='buttons', buttons=[
            dict(label='\u25b6 播放', method='animate',
                 args=[None, {'frame': {'duration': 120}, 'transition': {'duration': 60}}]),
        ])],
        sliders=[dict(steps=[dict(args=[[f.name]], label=f.name, method='animate') for f in frames])],
    )
    return fig

fig = animate_moment_frontiers()
fig.show()

In [ ]:
# Cell 4: 导入依赖
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from scipy.optimize import minimize
from scipy.stats import skew as scipy_skew, kurtosis as scipy_kurtosis

from shared.data_fetcher import get_etf_daily, get_stock_daily, get_index_daily
from shared.constants import (SECTOR_ETFS, DEFAULT_START, DEFAULT_END,
                               RISK_FREE_RATE, TRADING_DAYS_PER_YEAR, INITIAL_CAPITAL)
from shared.metrics import summary_table
from shared.visualization import (plot_equity_curve, plot_drawdown,
                                   plot_metrics_table, plot_cumulative_comparison)

np.random.seed(42)
print('All imports loaded successfully.')

In [ ]:
# Cell 5: 下载行业ETF数据
etf_names = list(SECTOR_ETFS.keys())
etf_codes = list(SECTOR_ETFS.values())

fallback_stocks = {
    '银行': '601398', '券商': '600030', '医药': '600276', '消费': '000858',
    '科技': '002415', '新能源': '300750', '军工': '600893', '地产': '001979',
}

close_dict = {}
for name, code in SECTOR_ETFS.items():
    df = get_etf_daily(code, DEFAULT_START, DEFAULT_END)
    if df is not None and len(df) > 100 and 'close' in df.columns:
        close_dict[name] = df['close']
    else:
        print(f'ETF {name}({code}) 失败, 使用备用个股')
        df = get_stock_daily(fallback_stocks[name], DEFAULT_START, DEFAULT_END)
        if df is not None and len(df) > 100 and 'close' in df.columns:
            close_dict[name] = df['close']

prices_df = pd.DataFrame(close_dict).sort_index().ffill().dropna()

bench_df = get_index_daily('sh000300', DEFAULT_START, DEFAULT_END)
bench_close = bench_df['close'] if len(bench_df) > 0 else None

print(f'数据区间: {prices_df.index[0].date()} ~ {prices_df.index[-1].date()}')
print(f'资产数量: {prices_df.shape[1]}, 交易日: {prices_df.shape[0]}')

In [ ]:
# Cell 6: 高阶矩计算 + 4-Moment优化函数

def portfolio_moments(w, returns_matrix):
    """
    计算组合的四阶矩
    w: 权重向量 (N,)
    returns_matrix: (T, N) 收益率矩阵
    返回: (均值, 方差, 偏度, 峰度)
    """
    port_returns = returns_matrix @ w
    mu = np.mean(port_returns) * TRADING_DAYS_PER_YEAR
    var = np.var(port_returns, ddof=1) * TRADING_DAYS_PER_YEAR
    skewness = scipy_skew(port_returns)  # 无量纲
    kurt = scipy_kurtosis(port_returns, fisher=True)  # 超额峰度 (正态=0)
    return mu, var, skewness, kurt


def optimize_2moment(returns_matrix, risk_aversion=2.5):
    """
    传统2阶矩 Mean-Variance 优化
    max w^T mu - (lambda/2) w^T Sigma w
    """
    n = returns_matrix.shape[1]
    mu = returns_matrix.mean(axis=0) * TRADING_DAYS_PER_YEAR
    cov = np.cov(returns_matrix.T) * TRADING_DAYS_PER_YEAR

    def neg_utility(w):
        return -(w @ mu - 0.5 * risk_aversion * w @ cov @ w)

    constraints = [{'type': 'eq', 'fun': lambda w: np.sum(w) - 1.0}]
    bounds = [(0, 0.4)] * n
    w0 = np.ones(n) / n
    res = minimize(neg_utility, w0, method='SLSQP', bounds=bounds, constraints=constraints)
    return res.x if res.success else w0


def optimize_4moment(returns_matrix, risk_aversion=2.5,
                     skew_pref=1.0, kurt_aversion=0.5):
    """
    4阶矩 Mean-Variance-Skewness-Kurtosis 优化
    max alpha1*mu - alpha2*var + alpha3*skew - alpha4*kurt
    """
    n = returns_matrix.shape[1]

    def neg_utility_4m(w):
        mu, var, skewness, kurt = portfolio_moments(w, returns_matrix)
        # 效用 = 收益 - 风险厌恶*方差 + 偏度偏好*偏度 - 峰度厌恶*峰度
        utility = mu - 0.5 * risk_aversion * var + skew_pref * skewness - kurt_aversion * kurt
        return -utility

    constraints = [{'type': 'eq', 'fun': lambda w: np.sum(w) - 1.0}]
    bounds = [(0, 0.4)] * n
    w0 = np.ones(n) / n

    # 多起点优化 (避免局部最优)
    best_w = w0.copy()
    best_val = neg_utility_4m(w0)

    for trial in range(5):
        if trial == 0:
            w_init = w0
        else:
            w_init = np.random.dirichlet(np.ones(n))
            w_init = np.clip(w_init, 0, 0.4)
            w_init /= w_init.sum()

        try:
            res = minimize(neg_utility_4m, w_init, method='SLSQP',
                           bounds=bounds, constraints=constraints,
                           options={'maxiter': 200})
            if res.success and res.fun < best_val:
                best_val = res.fun
                best_w = res.x
        except Exception:
            continue

    return best_w


# 测试
test_ret = prices_df.pct_change().dropna().iloc[:120].values
w_test = np.ones(test_ret.shape[1]) / test_ret.shape[1]
mu, var, sk, ku = portfolio_moments(w_test, test_ret)
print(f'等权组合四阶矩: 收益={mu:.2%}, 方差={var:.4f}, 偏度={sk:.3f}, 峰度={ku:.3f}')
print('\n4-Moment优化函数定义完成')

In [ ]:
# Cell 7: 滚动窗口回测 - 4-Moment vs 2-Moment

returns_df = prices_df.pct_change().dropna()
asset_names = list(prices_df.columns)
n_assets = len(asset_names)

lookback = 120
rebalance_freq = 20
risk_aversion = 2.5
skew_pref = 1.0
kurt_aversion = 0.5

dates = returns_df.index

# 4-Moment策略
port_ret_4m = pd.Series(0.0, index=dates)
weights_4m = np.ones(n_assets) / n_assets
weight_hist_4m = []
moments_hist = []  # 记录各期四阶矩

# 2-Moment策略
port_ret_2m = pd.Series(0.0, index=dates)
weights_2m = np.ones(n_assets) / n_assets

# 等权
port_ret_ew = pd.Series(0.0, index=dates)
weights_ew = np.ones(n_assets) / n_assets

rebal_dates = []

for i in range(lookback, len(dates)):
    daily_ret = returns_df.iloc[i].values
    port_ret_4m.iloc[i] = weights_4m @ daily_ret
    port_ret_2m.iloc[i] = weights_2m @ daily_ret
    port_ret_ew.iloc[i] = weights_ew @ daily_ret

    if (i - lookback) % rebalance_freq == 0:
        window_ret = returns_df.iloc[i-lookback:i].values

        # 4-Moment优化
        try:
            weights_4m = optimize_4moment(
                window_ret, risk_aversion, skew_pref, kurt_aversion
            )
        except Exception:
            weights_4m = np.ones(n_assets) / n_assets

        # 2-Moment优化
        try:
            weights_2m = optimize_2moment(window_ret, risk_aversion)
        except Exception:
            weights_2m = np.ones(n_assets) / n_assets

        # 记录四阶矩
        mu_4, var_4, sk_4, ku_4 = portfolio_moments(weights_4m, window_ret)
        mu_2, var_2, sk_2, ku_2 = portfolio_moments(weights_2m, window_ret)
        moments_hist.append({
            'date': dates[i],
            '4m_return': mu_4, '4m_var': var_4, '4m_skew': sk_4, '4m_kurt': ku_4,
            '2m_return': mu_2, '2m_var': var_2, '2m_skew': sk_2, '2m_kurt': ku_2,
        })

        # 交易成本
        if len(rebal_dates) > 0:
            port_ret_4m.iloc[i] -= 0.002
            port_ret_2m.iloc[i] -= 0.002

        rebal_dates.append(dates[i])
        weight_hist_4m.append(weights_4m.copy())

    if i % 100 == 0:
        print(f'  进度: {i}/{len(dates)}')

# 截取
port_ret_4m = port_ret_4m.iloc[lookback:]
port_ret_2m = port_ret_2m.iloc[lookback:]
port_ret_ew = port_ret_ew.iloc[lookback:]

equity_4m = INITIAL_CAPITAL * (1 + port_ret_4m).cumprod()
equity_2m = INITIAL_CAPITAL * (1 + port_ret_2m).cumprod()
equity_ew = INITIAL_CAPITAL * (1 + port_ret_ew).cumprod()

print(f'\n调仓次数: {len(rebal_dates)}')
print(f'4-Moment终端净值: {equity_4m.iloc[-1]:,.0f}')
print(f'2-Moment终端净值: {equity_2m.iloc[-1]:,.0f}')
print(f'等权终端净值:      {equity_ew.iloc[-1]:,.0f}')

In [ ]:
# Cell 8: 回测统计 + 高阶矩对比分析

bench_returns = None
if bench_close is not None:
    bench_close_aligned = bench_close.reindex(port_ret_4m.index).ffill()
    bench_returns = bench_close_aligned.pct_change().fillna(0)

metrics_4m = summary_table(port_ret_4m, equity_4m, bench_returns)
metrics_2m = summary_table(port_ret_2m, equity_2m, bench_returns)
metrics_ew = summary_table(port_ret_ew, equity_ew, bench_returns)

compare_keys = ['年化收益率', '年化波动率', '夏普比率', '最大回撤', 'Calmar比率']
print(f'{"指标":<12} {"4-Moment":<14} {"2-Moment":<14} {"等权":<14}')
print('-' * 56)
for k in compare_keys:
    print(f'{k:<12} {metrics_4m.get(k, "N/A"):<14} {metrics_2m.get(k, "N/A"):<14} {metrics_ew.get(k, "N/A"):<14}')

# 实际组合偏度/峰度对比
print('\n=== 组合收益分布高阶矩 ===')
for name, rets in [('4-Moment', port_ret_4m), ('2-Moment', port_ret_2m), ('等权', port_ret_ew)]:
    sk = scipy_skew(rets.dropna())
    ku = scipy_kurtosis(rets.dropna(), fisher=True)
    print(f'{name}: 偏度={sk:.3f}, 超额峰度={ku:.3f}')

# 偏度/峰度随时间的演变
if moments_hist:
    mom_df = pd.DataFrame(moments_hist).set_index('date')
    print(f'\n=== 平均组合矩 (样本内) ===')
    print(f'4-Moment: 偏度={mom_df["4m_skew"].mean():.3f}, 峰度={mom_df["4m_kurt"].mean():.3f}')
    print(f'2-Moment: 偏度={mom_df["2m_skew"].mean():.3f}, 峰度={mom_df["2m_kurt"].mean():.3f}')

In [ ]:
# Cell 9: 可视化
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams['font.sans-serif'] = ['PingFang SC', 'SimHei', 'Arial Unicode MS']
matplotlib.rcParams['axes.unicode_minus'] = False

# 1) 三策略净值对比
plot_cumulative_comparison(
    {'4-Moment (MVSK)': port_ret_4m, '2-Moment (MV)': port_ret_2m, '等权基准': port_ret_ew},
    title='4阶矩优化 vs 2阶矩优化 vs 等权'
)

# 2) 回撤对比
plot_drawdown(equity_4m, title='4-Moment策略 - 回撤')

# 3) 偏度/峰度时序演变
if moments_hist:
    mom_df = pd.DataFrame(moments_hist).set_index('date')
    fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

    axes[0].plot(mom_df.index, mom_df['4m_skew'], label='4-Moment', color='#2196F3', linewidth=1.5)
    axes[0].plot(mom_df.index, mom_df['2m_skew'], label='2-Moment', color='#F44336', linewidth=1.5, alpha=0.7)
    axes[0].axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    axes[0].set_title('组合偏度 (Skewness) 演变', fontsize=13, fontweight='bold')
    axes[0].set_ylabel('偏度')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(mom_df.index, mom_df['4m_kurt'], label='4-Moment', color='#2196F3', linewidth=1.5)
    axes[1].plot(mom_df.index, mom_df['2m_kurt'], label='2-Moment', color='#F44336', linewidth=1.5, alpha=0.7)
    axes[1].axhline(y=0, color='gray', linestyle='--', alpha=0.5, label='正态 (k=0)')
    axes[1].set_title('组合超额峰度 (Excess Kurtosis) 演变', fontsize=13, fontweight='bold')
    axes[1].set_ylabel('超额峰度')
    axes[1].set_xlabel('日期')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

# 4) 权重堆积图
weight_df = pd.DataFrame(weight_hist_4m, columns=asset_names,
                          index=rebal_dates[:len(weight_hist_4m)])
fig, ax = plt.subplots(figsize=(14, 5))
weight_df.plot.area(ax=ax, alpha=0.75, colormap='Set2')
ax.set_title('4-Moment策略 - 资产权重演变', fontsize=14, fontweight='bold')
ax.set_ylabel('权重')
ax.set_ylim(0, 1)
ax.legend(loc='center left', bbox_to_anchor=(1.0, 0.5), fontsize=9)
plt.tight_layout()
plt.show()

# 5) 绩效表
plot_metrics_table(metrics_4m, title='4-Moment (MVSK) 策略绩效指标')

## 结果分析

### 高阶矩的价值
1. **偏度偏好**: 4-Moment优化倾向于选择正偏度的资产组合，提高获取超额收益的概率
2. **峰度厌恶**: 减少了对高峰度资产的配置，降低了极端尾部事件的风险
3. 在市场非正态分布较显著时（如A股），高阶矩信息的加入尤为重要

### 2-Moment vs 4-Moment
- 2-Moment在正态分布假设下是最优的，但A股收益率呈现显著的尖峰厚尾特征
- 4-Moment优化牺牲了部分均值-方差效率，换取了更好的尾部风险管理
- 在极端下跌行情中，4-Moment策略的最大回撤通常更小

### 实际应用注意
- 高阶矩的估计需要更多样本，窗口期不宜太短
- 偏度/峰度参数的选择 ($\alpha_3, \alpha_4$) 需要根据投资者偏好调整
- 非凸优化问题需要多起点搜索，计算成本较高
- 可结合鲁棒统计方法 (如M-估计量) 减少高阶矩的估计误差